# 🤖 Building a RAG Chatbot with LangChain — Full Tutorial

**Corso di Programmazione — Prof.ssa Flora Amato**  
**Università degli Studi di Napoli Federico II — DIETI**

---

## 📌 What You Will Learn

In this notebook, we will build a **Retrieval-Augmented Generation (RAG)** chatbot step by step.  
By the end, you will understand:

1. **What RAG is** and why it matters
2. How to **load and split documents** into chunks
3. How to create **vector embeddings** of text
4. How to store and search vectors in a **vector store**
5. How to connect a **Large Language Model (LLM)** to retrieved context
6. How to build a **conversational chain** with memory
7. How to create a simple **chat UI** inside Colab

---

## 📖 Part 0 — What is RAG?

### The Problem
Large Language Models (LLMs) like GPT-4 or Claude are trained on vast amounts of data, but they have **two critical limitations**:

| Limitation | Example |
|---|---|
| **Knowledge cutoff** | The model doesn't know about events after its training date |
| **No access to private data** | The model can't read your company documents, PDFs, or databases |

### The Solution: RAG

**Retrieval-Augmented Generation** solves this by adding a *retrieval step* before generation:

```
User Question
     │
     ▼
┌─────────────┐
│  RETRIEVER   │ ──► Search your documents for relevant passages
└─────────────┘
     │
     ▼ (relevant context)
┌─────────────┐
│  GENERATOR   │ ──► LLM generates answer USING the retrieved context
│   (LLM)      │
└─────────────┘
     │
     ▼
  Answer grounded in YOUR data
```

### Why is this powerful?
- ✅ The LLM answers based on **your actual documents**
- ✅ Reduces **hallucinations** (making things up)
- ✅ No need to **retrain** the model
- ✅ Works with **any document type** (PDF, web pages, databases, etc.)

---

## 🏗️ Architecture Overview

Here is the full pipeline we will build:

```
                        ┌──────────────────────────────────────┐
                        │        INDEXING PHASE (offline)       │
                        │                                      │
  Documents ──► Load ──► Split into Chunks ──► Embed ──► Store │
  (PDF, txt,    (1)         (2)                 (3)     in DB  │
   web, ...)                                            (4)    │
                        └──────────────────────────────────────┘

                        ┌──────────────────────────────────────┐
                        │        QUERY PHASE (online)          │
                        │                                      │
  User Query ──► Embed ──► Search Vector DB ──► Retrieve Top-K │
     (5)         (6)           (7)               Chunks (8)    │
                        │                                      │
                        │  Prompt + Context ──► LLM ──► Answer │
                        │       (9)            (10)     (11)   │
                        └──────────────────────────────────────┘
```

We will build **each component** one at a time.

---

## ⚙️ Part 1 — Setup & Installation

First, we install all the libraries we need. Each one has a specific role:

| Library | Purpose |
|---|---|
| `langchain` | Core framework for building LLM applications |
| `langchain-community` | Community integrations (loaders, vector stores, etc.) |
| `langchain-openai` | OpenAI-specific LLM and embedding wrappers |
| `langchain-huggingface` | HuggingFace embeddings (free, no API key needed) |
| `chromadb` | Lightweight vector database (runs locally) |
| `pypdf` | For reading PDF files |
| `tiktoken` | OpenAI's tokenizer (used for token counting) |

In [ ]:
# 📦 Install all required packages
# The -q flag means "quiet" — less output during installation

!pip install -q langchain langchain-community langchain-openai langchain-huggingface
!pip install -q chromadb pypdf tiktoken
!pip install -q sentence-transformers

print("\n✅ All packages installed successfully!")

In [ ]:
# 🔑 API Key Configuration
# We need an OpenAI API key for the LLM (GPT-3.5 / GPT-4).
# If you don't have one, we'll also show a FREE alternative using HuggingFace.

import os
from getpass import getpass

# Option A: Use OpenAI (recommended for best results)
# Get your key at: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API Key (or press Enter to skip): ")

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY", "").strip())

if USE_OPENAI:
    print("✅ OpenAI API key set. We will use GPT-3.5-turbo as the LLM.")
else:
    print("⚠️ No OpenAI key provided. We will use HuggingFace (free) as a fallback.")
    print("   Note: Free models are slower and less capable, but great for learning!")

---

## 📄 Part 2 — Loading Documents

The first step in any RAG pipeline is to **load your data**.

LangChain provides **Document Loaders** for many formats:
- `PyPDFLoader` — for PDF files
- `TextLoader` — for plain text files
- `WebBaseLoader` — for web pages
- `CSVLoader` — for CSV files
- ... and many more!

Each loader returns a list of `Document` objects. A `Document` has two fields:
- `page_content` → the actual text
- `metadata` → information about the source (filename, page number, etc.)

### Let's create some sample documents
For this tutorial, we'll create sample text files about a fictional topic. In a real project, you would load your own PDFs, web pages, etc.

In [ ]:
# 📝 Let's create some sample documents to work with.
# In a real project, these could be your PDFs, web pages, or database records.

import os

os.makedirs("sample_docs", exist_ok=True)

# Document 1: About Python
with open("sample_docs/python_basics.txt", "w") as f:
    f.write("""Python Programming Language — A Comprehensive Overview

Python is a high-level, interpreted programming language created by Guido van Rossum
and first released in 1991. It emphasizes code readability with its notable use of
significant indentation. Python supports multiple programming paradigms, including
structured, object-oriented, and functional programming.

Key Features of Python:
Python uses dynamic typing and garbage collection. It has a large standard library
that is often described as having a "batteries included" philosophy. The language
provides constructs that enable clear programming on both small and large scales.

Data Types in Python:
Python has several built-in data types: integers (int), floating-point numbers (float),
strings (str), booleans (bool), lists, tuples, dictionaries (dict), and sets.
Lists are mutable ordered sequences, while tuples are immutable. Dictionaries store
key-value pairs and are extremely efficient for lookups.

Control Flow:
Python uses if/elif/else for conditional execution, for loops for iteration over
sequences, while loops for repeated execution, and try/except for error handling.
The for loop in Python iterates over items of any sequence (list, string, etc.)
rather than iterating over arithmetic progressions as in some other languages.

Functions in Python:
Functions are defined using the 'def' keyword. Python supports default arguments,
keyword arguments, *args (variable positional arguments), and **kwargs (variable
keyword arguments). Functions are first-class objects, meaning they can be passed
as arguments, returned from other functions, and assigned to variables.

Python is widely used in web development (Django, Flask), data science (pandas,
NumPy, scikit-learn), artificial intelligence (TensorFlow, PyTorch), automation,
and scripting. It is one of the most popular languages in the world.""")

# Document 2: About OOP in Python
with open("sample_docs/python_oop.txt", "w") as f:
    f.write("""Object-Oriented Programming (OOP) in Python

Object-Oriented Programming is a paradigm based on the concept of 'objects', which
contain data (attributes) and code (methods). Python has been an object-oriented
language since its inception. Everything in Python is an object.

Classes and Objects:
A class is a blueprint for creating objects. It defines attributes and methods that
the objects will have. You create a class using the 'class' keyword. An object is
an instance of a class. The __init__ method is the constructor, called when a new
object is created.

Example:
class Dog:
    def __init__(self, name, breed):
        self.name = name      # instance attribute
        self.breed = breed    # instance attribute

    def bark(self):
        return f"{self.name} says Woof!"

The Four Pillars of OOP:

1. Encapsulation: Bundling data and methods within a class, and restricting direct
   access to some components. In Python, we use underscore conventions: _protected
   and __private (name mangling).

2. Inheritance: Creating a new class (child) from an existing class (parent). The
   child inherits attributes and methods from the parent. Python supports multiple
   inheritance, where a class can inherit from multiple parent classes.

3. Polymorphism: The ability to use a common interface for different underlying forms.
   In Python, this is naturally achieved through duck typing — if an object has the
   required method, it can be used regardless of its class.

4. Abstraction: Hiding complex implementation details and showing only the necessary
   features. Python provides abstract base classes (ABC module) for this purpose.

Special Methods (Dunder Methods):
Python uses double-underscore methods like __str__, __repr__, __len__, __add__,
__eq__, etc. to define how objects behave with built-in operations. For example,
defining __str__ controls what happens when you call str() or print() on an object.

Decorators like @property, @staticmethod, and @classmethod provide additional
tools for structuring OOP code in Python.""")

# Document 3: About Error Handling
with open("sample_docs/python_errors.txt", "w") as f:
    f.write("""Error Handling and Exceptions in Python

Errors in Python are categorized into two types: syntax errors and exceptions.
Syntax errors occur when the parser detects an incorrect statement. Exceptions
occur during execution and can be handled gracefully.

The Try/Except Block:
The fundamental mechanism for handling exceptions is the try/except block.
Code that might raise an exception is placed in the try block, and the handling
code goes in the except block.

try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")
except TypeError as e:
    print(f"Type error: {e}")
else:
    print("No exception occurred")  # runs only if no exception
finally:
    print("This always runs")       # runs regardless

Common Built-in Exceptions:
- ValueError: Raised when a function receives an argument of the right type but
  inappropriate value.
- TypeError: Raised when an operation is applied to an object of inappropriate type.
- KeyError: Raised when a dictionary key is not found.
- IndexError: Raised when a sequence index is out of range.
- FileNotFoundError: Raised when trying to open a file that doesn't exist.
- AttributeError: Raised when an attribute reference or assignment fails.
- ImportError: Raised when an import statement fails to find the module.

Custom Exceptions:
You can create your own exceptions by inheriting from the Exception class:

class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw {amount}, balance is {balance}")

Best Practices:
1. Catch specific exceptions, not generic ones.
2. Use finally for cleanup operations (closing files, releasing resources).
3. Don't use exceptions for normal control flow.
4. Use context managers (with statement) when working with resources.
5. Log exceptions for debugging purposes.""")

print("✅ Sample documents created in 'sample_docs/' directory:")
for f in os.listdir("sample_docs"):
    size = os.path.getsize(f"sample_docs/{f}")
    print(f"   📄 {f} ({size} bytes)")

In [ ]:
# 📥 Step 2a: Load documents using LangChain's Document Loaders

from langchain_community.document_loaders import TextLoader, DirectoryLoader

# --- Method 1: Load a single file ---
single_loader = TextLoader("sample_docs/python_basics.txt")
single_doc = single_loader.load()

print("=" * 60)
print("📄 LOADING A SINGLE FILE")
print("=" * 60)
print(f"Type returned: {type(single_doc)}")
print(f"Number of documents: {len(single_doc)}")
print(f"\nDocument structure:")
print(f"  - page_content (first 200 chars): {single_doc[0].page_content[:200]}...")
print(f"  - metadata: {single_doc[0].metadata}")

# --- Method 2: Load all files from a directory ---
dir_loader = DirectoryLoader(
    "sample_docs/",       # directory path
    glob="**/*.txt",       # file pattern to match
    loader_cls=TextLoader  # which loader to use for each file
)
all_docs = dir_loader.load()

print(f"\n{'=' * 60}")
print("📂 LOADING AN ENTIRE DIRECTORY")
print("=" * 60)
print(f"Total documents loaded: {len(all_docs)}")
for i, doc in enumerate(all_docs):
    print(f"  📄 Doc {i}: {doc.metadata['source']} ({len(doc.page_content)} chars)")

In [ ]:
# 🌐 BONUS: Loading from the Web (WebBaseLoader)
# This is how you'd load a web page as a document.

# Uncomment the lines below to try it:

# !pip install -q beautifulsoup4
# from langchain_community.document_loaders import WebBaseLoader
# web_loader = WebBaseLoader("https://docs.python.org/3/tutorial/classes.html")
# web_docs = web_loader.load()
# print(f"Loaded {len(web_docs)} documents from the web")
# print(f"First 300 chars: {web_docs[0].page_content[:300]}")

# 📄 BONUS: Loading PDFs
# from langchain_community.document_loaders import PyPDFLoader
# pdf_loader = PyPDFLoader("your_file.pdf")
# pdf_docs = pdf_loader.load()  # Returns one Document per page!

print("💡 Tip: LangChain supports 100+ document loaders!")
print("   See: https://python.langchain.com/docs/integrations/document_loaders/")

---

## ✂️ Part 3 — Splitting Documents into Chunks

### Why do we need to split?

LLMs have a **limited context window** (e.g., 4096 tokens for GPT-3.5). We can't feed an entire book as context. Instead, we:

1. Split documents into **smaller chunks**
2. Find only the **most relevant chunks** for each question
3. Pass only those chunks to the LLM

### Key Parameters

| Parameter | Description | Typical Value |
|---|---|---|
| `chunk_size` | Maximum number of characters per chunk | 500–1500 |
| `chunk_overlap` | Characters shared between consecutive chunks | 50–200 |

### Why overlap?
Overlap ensures that information at the **boundary** between two chunks is not lost.

```
Without overlap:  [chunk 1          ] [chunk 2          ] [chunk 3          ]
                  ← info lost here → ← info lost here →

With overlap:     [chunk 1          ]
                         [chunk 2          ]
                                [chunk 3          ]
                  ← overlapping regions preserve context →
```

In [ ]:
# ✂️ Step 3: Split documents into chunks

from langchain.text_splitter import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter is the RECOMMENDED splitter.
# It tries to split on natural boundaries in this order:
# "\n\n" (paragraphs) → "\n" (lines) → " " (words) → "" (characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Max characters per chunk
    chunk_overlap=100,    # Overlap between consecutive chunks
    length_function=len,  # How to measure chunk size
    separators=["\n\n", "\n", " ", ""]  # Priority order for splitting
)

# Split all documents
chunks = text_splitter.split_documents(all_docs)

print(f"📊 Splitting Results:")
print(f"   Original documents: {len(all_docs)}")
print(f"   After splitting:    {len(chunks)} chunks")
print(f"\n{'=' * 60}")

# Let's examine the first few chunks
for i, chunk in enumerate(chunks[:4]):
    print(f"\n📦 CHUNK {i}")
    print(f"   Source: {chunk.metadata['source']}")
    print(f"   Length: {len(chunk.page_content)} characters")
    print(f"   Content preview: {chunk.page_content[:150]}...")
    print("-" * 40)

In [ ]:
# 🔬 Let's visualize the overlap between consecutive chunks from the same document

print("🔬 VISUALIZING CHUNK OVERLAP")
print("=" * 60)

# Find two consecutive chunks from the same source
for i in range(len(chunks) - 1):
    if chunks[i].metadata['source'] == chunks[i+1].metadata['source']:
        end_of_chunk_1 = chunks[i].page_content[-120:]
        start_of_chunk_2 = chunks[i+1].page_content[:120]

        print(f"\n📦 End of Chunk {i}:")
        print(f"   ...{end_of_chunk_1}")
        print(f"\n📦 Start of Chunk {i+1}:")
        print(f"   {start_of_chunk_2}...")
        print(f"\n👉 Notice: the text OVERLAPS! This preserves context at boundaries.")
        break

---

## 🧮 Part 4 — Embeddings: Turning Text into Vectors

### What are embeddings?

An **embedding** is a numerical representation (vector) of text that captures its **semantic meaning**.

```
"Python is a programming language"  →  [0.12, -0.45, 0.78, 0.03, ...] (1536 dimensions)
"Java is a coding language"         →  [0.11, -0.43, 0.76, 0.05, ...] (similar!)
"I love pizza"                      →  [0.89, 0.23, -0.56, 0.67, ...] (very different!)
```

### Why embeddings?
- Texts with **similar meaning** will have **similar vectors** (close in vector space)
- This lets us find relevant documents using **vector similarity search**
- We can compare the user's question embedding with all chunk embeddings

### How similarity works
We use **cosine similarity** to measure how close two vectors are:
- **1.0** = identical meaning
- **0.0** = completely unrelated
- **-1.0** = opposite meaning

In [ ]:
# 🧮 Step 4: Create an embedding model

# We'll use HuggingFace's free embedding model for everyone,
# regardless of whether they have an OpenAI key.
# This model runs LOCALLY — no API calls, no costs!

from langchain_huggingface import HuggingFaceEmbeddings

# 'all-MiniLM-L6-v2' is a small, fast, and effective model
# It produces 384-dimensional vectors
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},   # Use 'cuda' if you have a GPU
    encode_kwargs={"normalize_embeddings": True}  # Normalize for cosine similarity
)

print("✅ Embedding model loaded: all-MiniLM-L6-v2")
print("   This model runs locally — no API key needed!")

In [ ]:
# 🔬 Let's see embeddings in action!

import numpy as np

# Embed some example sentences
sentences = [
    "Python is a programming language",
    "Java is a coding language",
    "I love eating pizza on weekends",
    "Object-oriented programming uses classes and objects",
]

embeddings = embedding_model.embed_documents(sentences)

print("🔬 EXPLORING EMBEDDINGS")
print("=" * 60)
print(f"Each sentence becomes a vector of {len(embeddings[0])} dimensions")
print(f"\nExample — first 10 values of sentence 1:")
print(f"  {embeddings[0][:10]}")

# Calculate cosine similarity between all pairs
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(f"\n📐 COSINE SIMILARITY MATRIX")
print("-" * 60)
for i, s1 in enumerate(sentences):
    for j, s2 in enumerate(sentences):
        if j > i:
            sim = cosine_similarity(embeddings[i], embeddings[j])
            emoji = "🟢" if sim > 0.5 else "🟡" if sim > 0.3 else "🔴"
            print(f"  {emoji} {sim:.4f}  \"{s1[:40]}\"  ↔  \"{s2[:40]}\"")

print(f"\n💡 Notice: semantically similar sentences have HIGHER similarity scores!")

---

## 🗄️ Part 5 — Vector Store: Indexing Our Chunks

Now we need to **store** our embedded chunks so we can search them efficiently.

A **Vector Store** (or Vector Database) is a specialized database that:
1. Stores vectors alongside their original text and metadata
2. Performs fast **similarity search** (finding nearest vectors)

We'll use **ChromaDB**, which is:
- Free and open source
- Runs locally (no server needed)
- Perfect for learning and prototyping

Other popular options: Pinecone, Weaviate, Qdrant, FAISS, pgvector

In [ ]:
# 🗄️ Step 5: Create the Vector Store

from langchain_community.vectorstores import Chroma

# This single line does A LOT:
# 1. Takes each chunk's text
# 2. Creates an embedding vector for it
# 3. Stores text + vector + metadata in ChromaDB

vectorstore = Chroma.from_documents(
    documents=chunks,           # Our split document chunks
    embedding=embedding_model,  # The model to create embeddings
    collection_name="python_tutorial",  # Name for this collection
    persist_directory="./chroma_db"     # Where to save on disk
)

print(f"✅ Vector store created!")
print(f"   Total chunks indexed: {vectorstore._collection.count()}")
print(f"   Stored in: ./chroma_db/")

In [ ]:
# 🔍 Let's test the vector store with a similarity search!

query = "How do I define a class in Python?"

print(f"🔍 QUERY: \"{query}\"")
print("=" * 60)

# Search for the 3 most similar chunks
results = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results):
    print(f"\n📦 RESULT {i+1} (similarity score: {score:.4f})")
    print(f"   Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")
    print("-" * 40)

print("\n💡 Lower score = MORE similar (it's a distance metric in Chroma)")
print("   The retriever found the most relevant chunks about classes!")

---

## 🔍 Part 6 — The Retriever

A **Retriever** is LangChain's abstraction for searching a vector store.
It wraps the vector store and returns relevant `Document` objects.

Key parameter: **`k`** = number of chunks to retrieve (typically 3-5)

In [ ]:
# 🔍 Step 6: Create a Retriever from the vector store

retriever = vectorstore.as_retriever(
    search_type="similarity",  # Can also be "mmr" (Maximum Marginal Relevance)
    search_kwargs={"k": 3}     # Return top 3 most relevant chunks
)

# Test the retriever
query = "What are Python decorators?"
retrieved_docs = retriever.invoke(query)

print(f"🔍 Query: \"{query}\"")
print(f"   Retrieved {len(retrieved_docs)} chunks\n")

for i, doc in enumerate(retrieved_docs):
    print(f"📦 Chunk {i+1}: {doc.page_content[:150]}...")
    print()

---

## 🧠 Part 7 — The LLM (Large Language Model)

Now we set up the **brain** of our chatbot — the LLM that will generate answers.

We provide two options:
- **Option A**: OpenAI's GPT-3.5-turbo (requires API key, fast, great quality)
- **Option B**: HuggingFace free model (no API key, slower, good for learning)

In [ ]:
# 🧠 Step 7: Set up the LLM

if USE_OPENAI:
    # Option A: OpenAI
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model_name="gpt-3.5-turbo",
        temperature=0.3,  # Lower = more deterministic, higher = more creative
        max_tokens=1000
    )
    print("✅ Using OpenAI GPT-3.5-turbo")

else:
    # Option B: HuggingFace (free)
    from langchain_community.llms import HuggingFaceHub

    # You need a free HuggingFace token from: https://huggingface.co/settings/tokens
    hf_token = getpass("🔑 Enter your HuggingFace token (free at huggingface.co): ")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

    llm = HuggingFaceHub(
        repo_id="google/flan-t5-large",  # Free model, good for Q&A
        model_kwargs={"temperature": 0.3, "max_length": 512}
    )
    print("✅ Using HuggingFace flan-t5-large (free)")

# Quick test
response = llm.invoke("What is 2 + 2? Answer in one word.")
print(f"   Quick test: 2+2 = {response}")

---

## 🔗 Part 8 — Building the RAG Chain

Now we connect everything together! This is the **core of RAG**.

### The Prompt Template
We need to tell the LLM **how** to use the retrieved context. We do this with a **prompt template**:

```
You are a helpful assistant. Answer the question based ONLY on the following context:

{context}          ← The retrieved chunks go here

Question: {question}  ← The user's question goes here
Answer:
```

### Why "based ONLY on the context"?
This instruction helps **reduce hallucinations** — the LLM will stick to what's in the documents rather than making things up.

In [ ]:
# 🔗 Step 8a: Create the Prompt Template

from langchain.prompts import ChatPromptTemplate

RAG_PROMPT_TEMPLATE = """
You are a helpful teaching assistant for a Python programming course.
Answer the student's question based ONLY on the following context extracted
from the course materials.

Rules:
- If the answer is in the context, provide a clear and detailed explanation.
- If the answer is NOT in the context, say "I don't have information about that
  in the course materials" — do NOT make up an answer.
- Include code examples when relevant.
- Be encouraging and educational in tone.

Context:
{context}

Student's Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

print("✅ Prompt template created")
print("\n📋 Template variables: {context} and {question}")

In [ ]:
# 🔗 Step 8b: Build the RAG Chain using LCEL (LangChain Expression Language)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    """Combine retrieved documents into a single string for the prompt."""
    return "\n\n---\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# Build the chain using the | (pipe) operator — this is LCEL!
# Data flows left to right through each component:
#
#   1. Retriever fetches relevant chunks     → {context}
#   2. Question passes through unchanged     → {question}
#   3. Prompt template fills in the blanks   → formatted prompt
#   4. LLM generates the answer              → raw response
#   5. Output parser extracts the string     → clean answer

rag_chain = (
    {
        "context": retriever | format_docs,      # Step 1: Retrieve & format
        "question": RunnablePassthrough()         # Step 2: Pass question through
    }
    | prompt      # Step 3: Fill prompt template
    | llm         # Step 4: Generate answer
    | StrOutputParser()  # Step 5: Parse output to string
)

print("✅ RAG chain built!")
print("\n🔗 Chain flow: Question → Retrieve → Format → Prompt → LLM → Answer")

In [ ]:
# 🧪 Let's test the RAG chain!

test_questions = [
    "What are the four pillars of OOP in Python?",
    "How do I handle errors in Python?",
    "What data types does Python support?",
    "Explain what duck typing means.",
]

for question in test_questions:
    print(f"\n{'=' * 70}")
    print(f"❓ QUESTION: {question}")
    print("=" * 70)

    answer = rag_chain.invoke(question)
    print(f"\n🤖 ANSWER:\n{answer}")
    print("-" * 70)

In [ ]:
# 🔬 UNDER THE HOOD: Let's see what the retriever finds for a question
# This helps you debug and understand what context the LLM receives

debug_question = "How do I create a custom exception?"

print(f"🔬 DEBUGGING THE RAG PIPELINE")
print(f"   Question: \"{debug_question}\"")
print("=" * 70)

# Step 1: See what the retriever returns
retrieved = retriever.invoke(debug_question)
print(f"\n📥 STEP 1 — Retrieved {len(retrieved)} chunks:")
for i, doc in enumerate(retrieved):
    print(f"\n  Chunk {i+1} [{doc.metadata['source']}]:")
    print(f"  {doc.page_content[:200]}...")

# Step 2: See the formatted context
formatted_context = format_docs(retrieved)
print(f"\n📝 STEP 2 — Formatted context (first 500 chars):")
print(f"  {formatted_context[:500]}...")

# Step 3: See the filled prompt
filled_prompt = prompt.format(context=formatted_context, question=debug_question)
print(f"\n📋 STEP 3 — Filled prompt (first 600 chars):")
print(f"  {filled_prompt[:600]}...")

# Step 4: Get the answer
answer = rag_chain.invoke(debug_question)
print(f"\n🤖 STEP 4 — Final answer:")
print(f"  {answer}")

---

## 💬 Part 9 — Adding Conversation Memory

Our current chain is **stateless** — it doesn't remember previous messages.  
Let's add **conversation memory** so the chatbot can handle follow-up questions!

### How it works:
```
User: "What is inheritance?"
Bot:  "Inheritance is a mechanism where a child class..."

User: "Can you give me an example?"      ← Refers to previous answer!
Bot:  "Here's an example of inheritance:" ← Understands the context!
```

Without memory, the bot wouldn't know what "an example" refers to.

In [ ]:
# 💬 Step 9: Build a Conversational RAG Chain with Memory

from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

# Memory stores the last 5 exchanges (question-answer pairs)
memory = ConversationBufferWindowMemory(
    k=5,                            # Remember last 5 exchanges
    memory_key="chat_history",       # Key name used in the prompt
    return_messages=True,            # Return as Message objects
    output_key="answer"              # Which output to store
)

# Build the conversational chain
conversational_rag = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,     # Also return which chunks were used
    verbose=False                     # Set to True to see the internal steps
)

print("✅ Conversational RAG chain built with memory!")
print("   The chatbot now remembers the last 5 exchanges.")

In [ ]:
# 🧪 Test the conversational chain with follow-up questions

conversation = [
    "What is inheritance in Python?",
    "Can you show me an example?",           # ← Refers to "inheritance"
    "What about multiple inheritance?",       # ← Follow-up
    "How does error handling work?",          # ← New topic
    "What are the best practices for that?",  # ← Refers to "error handling"
]

for question in conversation:
    print(f"\n{'=' * 70}")
    print(f"👤 STUDENT: {question}")
    print("=" * 70)

    result = conversational_rag.invoke({"question": question})

    print(f"\n🤖 ASSISTANT: {result['answer']}")
    print(f"\n📚 Sources used: {[doc.metadata['source'] for doc in result['source_documents']]}")

print(f"\n\n📝 CONVERSATION HISTORY IN MEMORY:")
print("=" * 70)
for msg in memory.chat_memory.messages:
    role = "👤" if msg.type == "human" else "🤖"
    print(f"{role} {msg.content[:100]}..." if len(msg.content) > 100 else f"{role} {msg.content}")

---

## 🖥️ Part 10 — Interactive Chat Interface

Let's build a simple chat loop so you can interact with the chatbot directly!

In [ ]:
# 🖥️ Step 10: Interactive Chat Loop

# Reset memory for a fresh conversation
memory.clear()

print("🤖 Python Tutorial Chatbot")
print("=" * 50)
print("Ask me anything about Python programming!")
print("Based on: Python Basics, OOP, and Error Handling docs")
print("Type 'quit' to exit, 'history' to see chat history")
print("=" * 50)

while True:
    question = input("\n👤 You: ").strip()

    if not question:
        continue

    if question.lower() == 'quit':
        print("\n👋 Goodbye! Happy coding!")
        break

    if question.lower() == 'history':
        print("\n📜 Chat History:")
        for msg in memory.chat_memory.messages:
            role = "👤" if msg.type == "human" else "🤖"
            content = msg.content[:150] + "..." if len(msg.content) > 150 else msg.content
            print(f"  {role} {content}")
        continue

    try:
        result = conversational_rag.invoke({"question": question})
        print(f"\n🤖 Bot: {result['answer']}")

        # Show sources (optional)
        sources = set(doc.metadata['source'] for doc in result['source_documents'])
        print(f"\n   📚 Sources: {', '.join(sources)}")

    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("   Please try rephrasing your question.")

---

## 🎓 Part 11 — Summary & Key Takeaways

### What We Built

```
┌─────────────────────────────────────────────────────────┐
│                    RAG CHATBOT                          │
│                                                         │
│  📄 Documents → ✂️ Chunks → 🧮 Embeddings → 🗄️ VectorDB │
│                                                         │
│  👤 Question → 🔍 Retrieve → 📝 Prompt → 🧠 LLM → 💬 Answer │
│                                                         │
│  🔄 + Conversation Memory for follow-up questions       │
└─────────────────────────────────────────────────────────┘
```

### Component Summary

| Step | Component | LangChain Class | Purpose |
|---|---|---|---|
| 1 | Document Loader | `TextLoader`, `PyPDFLoader` | Load raw documents |
| 2 | Text Splitter | `RecursiveCharacterTextSplitter` | Break docs into chunks |
| 3 | Embedding Model | `HuggingFaceEmbeddings` | Convert text → vectors |
| 4 | Vector Store | `Chroma` | Store and search vectors |
| 5 | Retriever | `vectorstore.as_retriever()` | Find relevant chunks |
| 6 | Prompt Template | `ChatPromptTemplate` | Structure the LLM input |
| 7 | LLM | `ChatOpenAI` / `HuggingFaceHub` | Generate answers |
| 8 | Memory | `ConversationBufferWindowMemory` | Track conversation |
| 9 | Chain | `ConversationalRetrievalChain` | Connect everything |

### Key Concepts

1. **RAG = Retrieval + Generation** — Ground the LLM in your actual data
2. **Embeddings** capture semantic meaning — similar texts → similar vectors
3. **Chunk size matters** — too small = missing context, too large = noise
4. **Overlap prevents information loss** at chunk boundaries
5. **The prompt template** is crucial — it tells the LLM how to use the context
6. **Memory** enables multi-turn conversations with follow-up questions

---

## 🚀 Part 12 — Exercises for Students

### Exercise 1: Change the Documents 📄
Replace the sample text files with:
- Your own course notes (as .txt or .pdf files)
- A Wikipedia article about a topic you're studying
- The Python official documentation

### Exercise 2: Tune the Parameters ⚙️
Experiment with:
- `chunk_size`: Try 200, 500, 1000, 2000 — how does it affect answers?
- `chunk_overlap`: Try 0, 50, 100, 200 — what happens with no overlap?
- `k` (number of retrieved chunks): Try 1, 3, 5, 10
- `temperature`: Try 0.0, 0.5, 1.0 — how does creativity change?

### Exercise 3: Add Source Citations 📝
Modify the prompt template to make the LLM cite its sources:
```
Always end your answer with:
Sources: [list the source files used]
```

### Exercise 4: Try Different Embedding Models 🧮
Replace `all-MiniLM-L6-v2` with:
- `all-mpnet-base-v2` (768 dimensions, more accurate, slower)
- `paraphrase-multilingual-MiniLM-L12-v2` (for multilingual support!)

### Exercise 5: Add a Web Loader 🌐
Use `WebBaseLoader` to load a real web page and add it to your knowledge base.

### Exercise 6 (Advanced): Evaluate Your RAG 📊
Create a test set of 10 questions with known answers. Run each through the pipeline and measure:
- Does the retriever find the right chunks? (Retrieval quality)
- Does the LLM answer correctly? (Generation quality)
- Does it hallucinate? (Faithfulness)

---

## 📚 Further Reading

- [LangChain Documentation](https://python.langchain.com/docs/)
- [RAG Tutorial — LangChain](https://python.langchain.com/docs/tutorials/rag/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [HuggingFace Sentence Transformers](https://www.sbert.net/)
- [OpenAI API Reference](https://platform.openai.com/docs/api-reference)

### Advanced Topics to Explore
- **Hybrid Search**: Combine vector search with keyword search (BM25)
- **Re-ranking**: Use a cross-encoder to re-rank retrieved results
- **Agentic RAG**: Let the LLM decide when and how to retrieve
- **Multi-modal RAG**: Handle images, tables, and code alongside text
- **Evaluation Frameworks**: RAGAS, LangSmith for measuring RAG quality

---

**🎓 End of Tutorial — Prof.ssa Flora Amato, DIETI, UNINA**  
**Happy coding! 🐍**